In [176]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image, ImageEnhance, ImageFilter, ImageOps
import pandas as pd
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, log_loss, confusion_matrix, ConfusionMatrixDisplay
from scipy.ndimage import rotate, zoom, shift, convolve
import random
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, callbacks, Sequential
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Rescaling, RandomFlip, RandomRotation, RandomZoom, RandomTranslation, RandomContrast
from tensorflow.keras.utils import image_dataset_from_directory
import shutil

# Analisis exploratorio

In [177]:
data_dir = Path("data/PolyMNIST/MMNIST")
plt.style.use('default')
sns.set_palette("husl")

# Primero exploramos la estructura del dataset, para saber cuales directorios hay y la cantidad
# de informacion que tiene cada directorio

# Funcion para explorar directorios
def explore_directory_structure(data_dir):
    print("Estructura del dataset")    
    structure_info = {}
    # Como sabemos que hay directorios train y test los vamos a explorar
    for split in ['train', 'test']:
        split_path = data_dir / split
        if split_path.exists():
            print(f"\n{split.upper()}:")
            structure_info[split] = {}
            
            for modality_dir in sorted(split_path.iterdir()):
                if modality_dir.is_dir():
                    modality_name = modality_dir.name
                    png_files = list(modality_dir.glob("*.png"))
                    structure_info[split][modality_name] = len(png_files)
                    print(f"  └── {modality_name}: {len(png_files)} imágenes")
        else:
            print(f"No se encontró la carpeta: {split_path}")
    
    return structure_info

# Funcion para cargar muestras de las imagenes, se puede cambiar el numero de muestras, pero solo probare con una imagen por directorio
def load_sample_images(data_dir, n_samples=1):
    samples = defaultdict(lambda: defaultdict(list))
    image_info = defaultdict(lambda: defaultdict(list))
    
    for split in ['train', 'test']:
        split_path = data_dir / split
        if not split_path.exists():
            continue
            
        print(f"\nProcesando {split}...")
        
        for modality_dir in sorted(split_path.iterdir()):
            if modality_dir.is_dir():
                modality_name = modality_dir.name
                png_files = list(modality_dir.glob("*.png"))
                
                # Tomar una muestra aleatoria
                if len(png_files) > 0:
                    sample_files = np.random.choice(png_files, 
                                                  min(n_samples, len(png_files)), 
                                                  replace=False)
                    
                    for img_path in sample_files:
                        try:
                            img = Image.open(img_path)
                            img_array = np.array(img)
                            samples[split][modality_name].append(img_array)
                            
                            # Información de la imagen
                            image_info[split][modality_name].append({
                                'filename': img_path.name,
                                'size': img.size,
                                'mode': img.mode,
                                'shape': img_array.shape,
                                'dtype': img_array.dtype,
                                'min_val': img_array.min(),
                                'max_val': img_array.max(),
                                'mean_val': img_array.mean()
                            })
                        except Exception as e:
                            print(f"Error cargando {img_path}: {e}")
                
                print(f" {modality_name}: {len(samples[split][modality_name])} muestras cargadas")
    
    return  image_info
explore_directory_structure(data_dir)
load_sample_images(data_dir)


Estructura del dataset

TRAIN:
  └── m0: 60000 imágenes
  └── m1: 60000 imágenes
  └── m2: 60000 imágenes
  └── m3: 60000 imágenes
  └── m4: 60000 imágenes

TEST:
  └── m0: 10000 imágenes
  └── m1: 10000 imágenes
  └── m2: 10000 imágenes
  └── m3: 10000 imágenes
  └── m4: 10000 imágenes

Procesando train...
 m0: 1 muestras cargadas
 m1: 1 muestras cargadas
 m2: 1 muestras cargadas
 m3: 1 muestras cargadas
 m4: 1 muestras cargadas

Procesando test...
 m0: 1 muestras cargadas
 m1: 1 muestras cargadas
 m2: 1 muestras cargadas
 m3: 1 muestras cargadas
 m4: 1 muestras cargadas


defaultdict(<function __main__.load_sample_images.<locals>.<lambda>()>,
            {'train': defaultdict(list,
                         {'m0': [{'filename': '5531.0.png',
                            'size': (28, 28),
                            'mode': 'RGB',
                            'shape': (28, 28, 3),
                            'dtype': dtype('uint8'),
                            'min_val': 13,
                            'max_val': 244,
                            'mean_val': 168.00722789115648}],
                          'm1': [{'filename': '1042.8.png',
                            'size': (28, 28),
                            'mode': 'RGB',
                            'shape': (28, 28, 3),
                            'dtype': dtype('uint8'),
                            'min_val': 35,
                            'max_val': 219,
                            'mean_val': 113.63690476190476}],
                          'm2': [{'filename': '5602.9.png',
                          

# Transformaciones a imagenes

In [178]:
# Como el lab pide de avances hasta el paso 4, por el momento solo se dejaron definidas las transformaciones que vamos  a usar para
# entrenar en la proxima entrega el modelo

# Ahora las transformaciones se haran usando las funciones que trae tensorflow, y seran aleatorias para cada batch

data_augmentation = Sequential([
    RandomFlip("horizontal"),
    RandomRotation(0.1),            # ±10% de rotación (~±15 grados)
    RandomZoom(0.1),
    RandomTranslation(0.1, 0.1),    # 10% en ambos ejes
    RandomContrast(0.1)
])

# Para el KNN vamos a definir funciones que actuen de manera similar a las de tensorflow

def rotate_image(image, angle_range=(-15, 15)):
    angle = np.random.uniform(*angle_range)
    return rotate(image, angle, reshape=False, mode='nearest')

def scale_image(image, zoom_range=(0.9, 1.1)):
    zoom_factor = np.random.uniform(*zoom_range)
    if len(image.shape) == 3:
        zoom_factors = [zoom_factor, zoom_factor, 1]
    else:
        zoom_factors = zoom_factor
    return zoom(image, zoom_factors, mode='nearest')

def add_gaussian_noise(image, noise_level=0.1):
    noise = np.random.normal(0, noise_level * 255, image.shape)
    result = image.astype(np.float32) + noise
    return np.clip(result, 0, 255).astype(image.dtype)

def blur_image(image, size=3):
    kernel = np.ones((size, size)) / (size * size)
    if len(image.shape) == 3:
        result = np.zeros_like(image)
        for c in range(image.shape[2]):
            result[:, :, c] = convolve(image[:, :, c], kernel, mode='nearest')
    else:
        result = convolve(image, kernel, mode='nearest')
    return result

# Y que la aplicacion de la transformacion tambien sea random
def apply_random_transform(image):
    transforms = [rotate_image, scale_image, add_gaussian_noise, blur_image]
    num_transforms = np.random.randint(1, len(transforms) + 1)
    chosen = np.random.choice(transforms, num_transforms, replace=False)
    transformed = image
    for t in chosen:
        transformed = t(transformed)
    return transformed

# Modelos de CNN

In [179]:
# Esto es un codigo que sirve para probar si funciona la gpu (el proyecto corre en un dockerfile con cuda)
print("GPU disponible:", tf.config.list_physical_devices('GPU'))
if tf.config.list_physical_devices('GPU'):
    print("Configurando GPU...")
    tf.config.experimental.set_memory_growth(tf.config.list_physical_devices('GPU')[0], True)


GPU disponible: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Configurando GPU...


In [180]:
combined_train = data_dir / "train_combined"
combined_test = data_dir / "test_combined"

img_size = (28, 28)
batch_size = 256
val_split = 0.2
seed = 42

# Funcion para ayudar a tranformar el futuro train_ds y val_ds para el knn en un array de numpy
def dataset_to_numpy(ds):
    images = []
    labels = []
    for batch_x, batch_y in ds:
        images.append(batch_x.numpy())
        labels.append(batch_y.numpy())
    X = np.concatenate(images, axis=0)
    y = np.concatenate(labels, axis=0)
    return X, y

# organizar las imágenes en carpetas por clase
def reorganizar_por_clase(origen_dir, destino_dir):
    destino_dir.mkdir(parents=True, exist_ok=True)
    for modalidad_dir in sorted(origen_dir.glob("m*")):
        for img_path in modalidad_dir.glob("*.png"):
            clase = img_path.stem.split(".")[1]  # Extraer la clase desde el nombre
            destino_clase = destino_dir / clase
            destino_clase.mkdir(parents=True, exist_ok=True)
            new_name = f"{modalidad_dir.name}_{img_path.name}"  # Evitar sobrescritura
            shutil.copy(img_path, destino_clase / new_name)

# IMPORTANTE ---- NO IGNORAR ---- Ejecutar solo una vez (comenta después si ya están combinadas)
# si ya corriste esto 2 veces, borra las carpetas del contenedor, aunque como usa volumen se puede desde el host asi no se duplican datos

#reorganizar_por_clase(data_dir / "train", combined_train)
#reorganizar_por_clase(data_dir / "test", combined_test)

# cargar datasets directamente desde los directorios combinados
train_ds = image_dataset_from_directory(
    combined_train,
    validation_split=val_split,
    subset="training",
    seed=seed,
    image_size=img_size,
    batch_size=batch_size,
    color_mode="grayscale"
)

train_ds_augmented = train_ds.map(
    lambda x, y: (data_augmentation(x, training=True), y),
    num_parallel_calls=tf.data.AUTOTUNE
)

val_ds = image_dataset_from_directory(
    combined_train,
    validation_split=val_split,
    subset="validation",
    seed=seed,
    image_size=img_size,
    batch_size=batch_size,
    color_mode="grayscale"
)

test_ds = image_dataset_from_directory(
    combined_test,
    image_size=img_size,
    batch_size=batch_size,
    color_mode="grayscale",
    shuffle=False
)

# Normalizar
class_names = train_ds.class_names
normalization_layer = Rescaling(1. / 255)
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))
test_ds = test_ds.map(lambda x, y: (normalization_layer(x), y))

# Prints de verificacion 
print(f"Clases detectadas: {class_names}")
print(f"Número de clases: {len(class_names)}")

Found 300000 files belonging to 10 classes.
Using 240000 files for training.
Found 300000 files belonging to 10 classes.
Using 60000 files for validation.
Found 50000 files belonging to 10 classes.
Clases detectadas: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
Número de clases: 10


## Modelos CNN

In [181]:
def cnn_1():
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
        layers.MaxPooling2D(2, 2),
        
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D(2, 2),
        
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax') # Como tenemos 10 digitos, necesitamos algo que nos permita clasificar mas clases.
    ])
    opt = keras.optimizers.Adam(learning_rate=0.0005)
    model.compile(optimizer=opt, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model
    
def cnn_2():
    model = keras.Sequential([
        layers.Conv2D(32, (3,3), activation='relu', padding='same', kernel_initializer='he_uniform', input_shape=(28,28,1)),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(64, (3,3), activation='relu', padding='same', kernel_initializer='he_uniform'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(128, (3,3), activation='relu', padding='same', kernel_initializer='he_uniform'),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation='relu', kernel_initializer='he_uniform'),
        layers.Dropout(0.3),
        layers.Dense(10, activation='softmax')
    ])

    opt = keras.optimizers.Adam(learning_rate=0.0003)
    model.compile(optimizer=opt, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# RNN con MLP
def build_mlp():
    model = models.Sequential([
        layers.Flatten(input_shape=(28, 28, 1)),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.4),
        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        layers.Dense(10, activation='softmax')
    ])
    opt = keras.optimizers.Adam(learning_rate=0.001)
    model.compile(optimizer=opt, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def dataset_to_numpy_transformed(ds):
    images = []
    labels = []
    for batch_x, batch_y in ds:
        batch_x_np = batch_x.numpy()
        batch_y_np = batch_y.numpy()
        for i in range(batch_x_np.shape[0]):
            img = batch_x_np[i]
            # Denormalizar (está entre 0 y 1)
            img = (img * 255).astype(np.uint8)
            # Aplicar transformaciones
            img_t = apply_random_transform(img)
            # Volver a normalizar (opcional, depende si querés normalizar tú después)
            img_t = img_t.astype(np.float32) / 255.0
            images.append(img_t)
            labels.append(batch_y_np[i])
    X = np.stack(images)
    y = np.array(labels)
    return X, y

# KNN - datos sin alteracion de imagen
X_train, y_train = dataset_to_numpy(train_ds)
X_val, y_val = dataset_to_numpy(val_ds)

X_train_flat = X_train.reshape((X_train.shape[0], -1))
X_val_flat = X_val.reshape((X_val.shape[0], -1))

scaler = MinMaxScaler()
X_train_flat = scaler.fit_transform(X_train_flat)
X_val_flat = scaler.transform(X_val_flat)

knn = KNeighborsClassifier(n_neighbors=3, n_jobs=-1)
knn.fit(X_train_flat, y_train)
y_pred_labels = knn.predict(X_val_flat)
y_pred_probs = knn.predict_proba(X_val_flat)

accuracy = accuracy_score(y_val, y_pred_labels)
loss = log_loss(y_val, y_pred_probs)

# KNN - con imagenes transformadas (blur, zoom y todo eso)
X_train_t, y_train_t = dataset_to_numpy_transformed(train_ds_augmented)
X_val, y_val = dataset_to_numpy(val_ds)

X_train_flat = X_train.reshape((X_train.shape[0], -1))
X_val_flat = X_val.reshape((X_val.shape[0], -1))

scaler = MinMaxScaler()
X_train_flat = scaler.fit_transform(X_train_flat)
X_val_flat = scaler.transform(X_val_flat)

knn = KNeighborsClassifier(n_neighbors=3, n_jobs=-1)
knn.fit(X_train_flat, y_train)
y_pred_labels = knn.predict(X_val_flat)
y_pred_probs = knn.predict_proba(X_val_flat)

accuracy_a = accuracy_score(y_val, y_pred_labels)
loss_a = log_loss(y_val, y_pred_probs)

history_knn = {
    'val_accuracy': [accuracy],
    'val_loss': [loss]
}

# Callbacks
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-7,
    verbose=1
)

callbacks = [early_stopping, reduce_lr]


# Modelos sin transformaciones
mlp_model = build_mlp()
cnn1 = cnn_1()
cnn2 = cnn_2()

# Modelos con transformaciones
mlp_model_aug = build_mlp()
cnn1_aug = cnn_1()
cnn2_aug = cnn_2()


print("\nModelos SIN transformacion de imagen")
print("\nCNN 1")
history_cnn1 = cnn1.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=callbacks)
print("\nCNN 2")
history_cnn2 = cnn2.fit(train_ds, validation_data=val_ds, epochs=20, callbacks=callbacks)
print("\nMLP")
history_mlp = mlp_model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=callbacks)

print("\nModelos CON transformacion de imagen")
print("\nCNN 1")
history_cnn1_aug = cnn1_aug.fit(train_ds_augmented, validation_data=val_ds, epochs=15, callbacks=callbacks)
print("\nCNN 2")
history_cnn2_aug = cnn2_aug.fit(train_ds_augmented, validation_data=val_ds, epochs=20, callbacks=callbacks)
print("\nMLP")
history_mlp_aug = mlp_model_aug.fit(train_ds_augmented, validation_data=val_ds, epochs=15, callbacks=callbacks)

2025-08-03 21:20:52.500562: I tensorflow/core/framework/local_rendezvous.cc:421] Local rendezvous recv item cancelled. Key hash: 4800284434214545957
2025-08-03 21:20:52.500587: I tensorflow/core/framework/local_rendezvous.cc:421] Local rendezvous recv item cancelled. Key hash: 7796743027477537597
2025-08-03 21:20:52.500594: I tensorflow/core/framework/local_rendezvous.cc:421] Local rendezvous recv item cancelled. Key hash: 4553176708754522190
2025-08-03 21:20:52.500599: I tensorflow/core/framework/local_rendezvous.cc:421] Local rendezvous recv item cancelled. Key hash: 15528645727747441142
2025-08-03 21:20:52.500604: I tensorflow/core/framework/local_rendezvous.cc:421] Local rendezvous recv item cancelled. Key hash: 9912308506275328424
2025-08-03 21:20:52.500616: I tensorflow/core/framework/local_rendezvous.cc:421] Local rendezvous recv item cancelled. Key hash: 2447633419940438839
2025-08-03 21:20:52.500620: I tensorflow/core/framework/local_rendezvous.cc:421] Local rendezvous recv it

NameError: name 'convolve' is not defined

# Evaluar los modelos

In [ ]:
def evaluar(model, nombre, test_ds):
    loss, acc = model.evaluate(test_ds, verbose=0)
    print(f"{nombre} - Test Accuracy: {acc:.4f} | Loss: {loss:.4f}")

def evaluar_knn(nombre, accuracy, loss):
    print(f"{nombre} - Val Accuracy: {accuracy:.4f} | Log Loss: {loss:.4f}")

def plot_metrics(history, title="Modelo"):
    epochs = range(1, len(history.history['accuracy']) + 1)
    
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))

    # Accuracy
    ax[0].plot(epochs, history.history['accuracy'], label='Train Accuracy')
    ax[0].plot(epochs, history.history['val_accuracy'], label='Val Accuracy')
    ax[0].set_title(f'{title} - Accuracy')
    ax[0].set_xlabel('Epochs')
    ax[0].set_ylabel('Accuracy')
    ax[0].legend()
    ax[0].grid(True)

    # Loss
    ax[1].plot(epochs, history.history['loss'], label='Train Loss')
    ax[1].plot(epochs, history.history['val_loss'], label='Val Loss')
    ax[1].set_title(f'{title} - Loss')
    ax[1].set_xlabel('Epochs')
    ax[1].set_ylabel('Loss')
    ax[1].legend()
    ax[1].grid(True)

    plt.suptitle(f'{title} - Training Metrics', fontsize=14)
    plt.tight_layout()
    plt.show()

def plot_knn_results(accuracy, loss, title="KNN - k=3"):
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))

    ax[0].bar(['Accuracy'], [accuracy], color='skyblue')
    ax[0].set_ylim(0, 1)
    ax[0].set_title('Validation Accuracy')

    ax[1].bar(['Log Loss'], [loss], color='salmon')
    ax[1].set_ylim(0, 2)  # el log loss puede variar más
    ax[1].set_title('Validation Log Loss')

    fig.suptitle(title)
    plt.show()
    
print("\n Sin transformacion imagen")
evaluar(cnn1, "CNN 1", test_ds)
evaluar(cnn2, "CNN 2", test_ds)
evaluar(mlp_model, "MLP", test_ds)
evaluar_knn("KNN", accuracy, loss)

plot_metrics(history_cnn1, "CNN 1")
plot_metrics(history_cnn2, "CNN 2")
plot_metrics(history_mlp, "MLP")

print("\n Con transformacion imagen")
evaluar(cnn1_aug, "CNN 1 - T", test_ds)
evaluar(cnn2_aug, "CNN 2 - T", test_ds)
evaluar(mlp_model_aug, "MLP - T", test_ds)

plot_metrics(history_cnn1_aug, "CNN 1 - T")
plot_metrics(history_cnn2_aug, "CNN 2 - T")
plot_metrics(history_mlp_aug, "MLP - T")

# KNN
plot_knn_results(accuracy, loss)

# Evaluando numeros hecho a mano